# PrometheusNet training on PUMA

Colab Pro entry point for the instance-aware multitask architecture. Tissue is trained as semantic segmentation; nuclei are trained as center-based instances and evaluated with centroid F1.

## 1. Environment

In [ ]:
import importlib
import os
import sys
from pathlib import Path

REPO_DIR = '/content/prometheus'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

In [ ]:
%cd {REPO_DIR}
%pip install -q -r requirements.txt

In [ ]:
# Prefer this checkout over any unrelated third-party package named `prometheus`.
SOURCE_DIR = str(Path(REPO_DIR, 'src').resolve())
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)

sys.path.insert(0, SOURCE_DIR)
loaded = sys.modules.get('prometheus')
loaded_file = getattr(loaded, '__file__', None) if loaded is not None else None
if loaded is not None and (loaded_file is None or SOURCE_DIR not in str(Path(loaded_file).resolve())):
    for module_name in [name for name in sys.modules if name == 'prometheus' or name.startswith('prometheus.')]:
        del sys.modules[module_name]

In [ ]:
importlib.invalidate_caches()
from prometheus import api as prometheus_api

print('Prometheus API:', Path(prometheus_api.__file__).resolve())
!git rev-parse --short HEAD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU is required. In Colab select Runtime > Change runtime type > GPU, then rerun.')

print(f'PyTorch {torch.__version__}; CUDA {torch.version.cuda}')
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, torch.cuda.get_device_name(index), f'{props.total_memory / 2**30:.1f} GiB')

## 2. Resolved configuration

In [ ]:
from dataclasses import asdict
from pprint import pprint
from pathlib import Path
from prometheus.api import build_criterion, build_datamodule, build_model, build_trainer, load_config

CONFIG_PATH = 'configs/experiment/baseline_multitask.toml'
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/PUMA/dataset_PUMA')
RUN_DIR = Path('/content/drive/MyDrive/prometheus_runs/baseline_multitask_v1')

if not DRIVE_DATASET_DIR.is_dir():
    raise FileNotFoundError(f'PUMA dataset not found: {DRIVE_DATASET_DIR}')

RUN_DIR.mkdir(parents=True, exist_ok=True)
config = load_config(CONFIG_PATH)
config.data.root = str(DRIVE_DATASET_DIR)
config.data.split_manifest = str(RUN_DIR / 'puma_split.json')
config.paths.run_dir = str(RUN_DIR)

# --- Hardware overrides for a single large-VRAM GPU (A6000 48GB / A100 / H100) ---
# The dataset is small (~205 pairs -> ~185 train / ~20 val at validation_fraction=0.1),
# so the binding constraint is the NUMBER OF OPTIMIZER STEPS PER EPOCH, not VRAM.
# steps/epoch = ceil(185 / batch_size): batch 8 -> ~24, batch 16 -> ~12, batch 32 -> ~6.
# We use a real batch of 16 (no gradient accumulation): it fills most of the GPU at
# 1024px while keeping ~12 steps/epoch. Push to 24-32 only if metrics are still healthy.
config.trainer.batch_size = 16
config.trainer.gradient_accumulation = 1  # not needed on a big GPU; real batch = effective batch
config.trainer.num_workers = 8            # match to available CPU cores (drop to 4 on Colab Pro)
config.trainer.amp = True                 # mixed precision: less VRAM + faster, keep on

# The trainer applies NO automatic LR scaling and its cosine schedule steps PER EPOCH,
# so the LR must be set by hand for the larger effective batch (baseline 2e-4 @ batch 4).
config.optimizer.lr = 3e-4                # gentler than sqrt-scaled 4e-4: fine-tuning a pretrained encoder
config.trainer.min_lr = 1e-6              # must stay <= optimizer.lr

# --- Small-data quality levers (205 pairs total) ---------------------------------
# 1) Pretrained encoder: seed the ConvNeXt-V2 backbone from ImageNet instead of random
#    init, via build_model(..., pretrained=USE_PRETRAINED) below. Biggest single win.
USE_PRETRAINED = True
# 2) Weight EMA: evaluate/checkpoint a moving average of the weights; stabilizes the
#    noisy ~20-image validation and generalizes better. 0 disables it.
config.trainer.ema_decay = 0.999
# 3) Stronger regularization to fight overfitting on a tiny set.
config.optimizer.weight_decay = 0.05
config.model.drop_path_rate = 0.2
# 4) Stain/photometric augmentation is now applied by default in the train transform.
# 5) Class-weighted loss (inverse frequency) for tissue + nuclei — always on to fight
#    the majority-class collapse on this imbalanced dataset.
config.loss.class_weighting = True

config.validate()

effective_batch = config.trainer.batch_size * config.trainer.gradient_accumulation
print(f'effective_batch={effective_batch}  lr={config.optimizer.lr}  '
      f'ema={config.trainer.ema_decay}  pretrained={USE_PRETRAINED}')
pprint(asdict(config))

## 3. Data and one-batch smoke test

In [ ]:
from prometheus.data.puma.audit import audit_puma_dataset

audit = audit_puma_dataset(config.data.root)
print('samples', audit['sample_count'])
if audit['errors']:
    raise ValueError(f"Dataset audit failed: {audit['errors'][:5]}")

train_loader, validation_loader = build_datamodule(config)
batch = next(iter(train_loader))

print('images', tuple(batch.images.shape))
print('tissue', tuple(batch.tissue.mask.shape))
print('instances/image', [len(target.labels) for target in batch.nuclei])

### Visual inspection after normalization

The preview rescales the normalized tensor only for display. The model still receives the normalized tensor unchanged.

In [ ]:
from prometheus.visualization import visualize_multitask_batch

visualize_multitask_batch(batch, index=0, show_boxes=True)

In [ ]:
import gc

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats(device)

# pretrained=USE_PRETRAINED downloads ImageNet ConvNeXt-V2 weights and prints how many
# encoder tensors were loaded; this also validates the pretrained mapping early.
model = build_model(config, pretrained=USE_PRETRAINED).to(device)
batch = batch.to(device)
criterion = build_criterion(config)

with torch.amp.autocast('cuda', enabled=config.trainer.amp):
    output = model(batch.images)
    losses = criterion(output, batch)

losses['total'].backward()

print({name: round(value.detach().item(), 5) for name, value in losses.items()})

# VRAM probe: this ran a real fwd+bwd on the configured batch size, so peak here ~= the
# per-step training memory (training only adds a small, fixed AdamW optimizer state on top).
# Use the reported headroom to decide whether to raise config.trainer.batch_size above.
peak = torch.cuda.max_memory_allocated(device) / 2**30
total = torch.cuda.get_device_properties(device).total_memory / 2**30
print(f'batch_size={config.trainer.batch_size}  peak_vram={peak:.2f} GiB / {total:.1f} GiB '
      f'({100 * peak / total:.0f}% used, {total - peak:.1f} GiB free)')

del batch, output, losses, criterion, model

In [ ]:
gc.collect()
torch.cuda.empty_cache()

## 4. Train or resume

In [ ]:
# Only seed from ImageNet on a fresh run; when resuming, the checkpoint weights win,
# so skip the (redundant) pretrained download.
last_checkpoint = Path(config.paths.run_dir) / 'last.ckpt'
resume_from = last_checkpoint if last_checkpoint.exists() else None

model = build_model(config, pretrained=USE_PRETRAINED and resume_from is None)
trainer = build_trainer(config, model=model, datamodule=(train_loader, validation_loader), device=device)
metrics = trainer.fit(resume_from=resume_from)

print(metrics)

## 5. Exact validation and artifacts

In [ ]:
from prometheus.engine import evaluate_multitask, load_engine_checkpoint, select_inference_state

best_path = Path(config.paths.run_dir) / 'best_primary.ckpt'

# Load the trained weights directly (no optimizer/RNG restore needed just to evaluate).
# select_inference_state prefers the EMA weights, which is what drove checkpoint selection.
checkpoint = load_engine_checkpoint(best_path, device)
eval_model = build_model(config).to(device)          # pretrained not needed; weights are overwritten
eval_model.load_state_dict(select_inference_state(checkpoint))

result = evaluate_multitask(
    eval_model, validation_loader, build_criterion(config), device,
    config.model.nuclei_feature_stride, config.evaluation.nuclei_radius_px,
    config.postprocess.confidence_threshold, config.postprocess.max_detections,
    config.postprocess.local_max_kernel, tta=True,
)

print('tissue Dice:', result.tissue_dice)
print('nuclei centroid F1:', result.nuclei_macro_f1)
print('epoch:', checkpoint['epoch'], '| checkpoint:', best_path)

## 5b. Visual sanity check — predictions vs ground truth

Runs the best (EMA) checkpoint on one validation image and overlays results so you can eyeball quality:
top row compares the **tissue** map (GT vs prediction), bottom row compares **nuclei** (centroid `x` + box, colored by class). Change `SAMPLE_INDEX` to inspect other images in the batch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from prometheus.engine import load_engine_checkpoint, select_inference_state
from prometheus.inference import decode_nuclei
from prometheus.domain import NucleusClass
from prometheus.data.puma import NUCLEI_CLASSES, TISSUE_CLASSES

SAMPLE_INDEX = 0  # which image in the validation batch to inspect

# Load the best (EMA) weights, independent of earlier cells so this can run on its own.
viz_model = build_model(config).to(device)
viz_ckpt = load_engine_checkpoint(Path(config.paths.run_dir) / 'best_primary.ckpt', device)
viz_model.load_state_dict(select_inference_state(viz_ckpt))
viz_model.eval()

batch = next(iter(validation_loader)).to(device)
with torch.no_grad():
    output = viz_model(batch.images)

# Predictions in MODEL space (metadata=None) so they overlay the 1024px letterboxed input.
detections = decode_nuclei(
    output, metadata=None,
    stride=config.model.nuclei_feature_stride,
    threshold=config.postprocess.confidence_threshold,
    max_detections=config.postprocess.max_detections,
    local_max_kernel=config.postprocess.local_max_kernel,
)[SAMPLE_INDEX]

nucleus_classes = list(NucleusClass)


def _preview(image_tensor):
    """Contrast-safe RGB preview of a normalized CHW tensor."""
    array = image_tensor.detach().float().cpu().permute(1, 2, 0).numpy()
    out = np.zeros_like(array)
    for channel in range(array.shape[-1]):
        low, high = np.percentile(array[..., channel], (1, 99))
        out[..., channel] = np.clip((array[..., channel] - low) / max(high - low, 1e-8), 0.0, 1.0)
    return out


def _draw_nuclei(axis, centroids, labels, boxes, confidences=None):
    palette = plt.get_cmap('tab10')
    if len(centroids):
        axis.scatter(centroids[:, 0], centroids[:, 1], c=labels, cmap='tab10',
                     vmin=0, vmax=len(NUCLEI_CLASSES) - 1, s=16, marker='x', linewidths=1.0)
        for i, (box, label) in enumerate(zip(boxes, labels)):
            x0, y0, x1, y1 = box
            axis.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                     edgecolor=palette(int(label) % 10), linewidth=0.7, alpha=0.8))


image = _preview(batch.images[SAMPLE_INDEX])
tissue_gt = batch.tissue.mask[SAMPLE_INDEX].cpu().numpy()
tissue_pred = output.tissue_logits[SAMPLE_INDEX].argmax(0).cpu().numpy()

gt = batch.nuclei[SAMPLE_INDEX]
gt_centroids = gt.centroids.cpu().numpy()
gt_labels = gt.labels.cpu().numpy()
gt_boxes = gt.boxes.cpu().numpy()

pred_centroids = np.array([d.centroid for d in detections], dtype=np.float32).reshape(-1, 2)
pred_labels = np.array([nucleus_classes.index(d.label) for d in detections], dtype=np.int64)
pred_boxes = np.array([d.box_xyxy for d in detections], dtype=np.float32).reshape(-1, 4)

figure, axes = plt.subplots(2, 3, figsize=(19, 12))
axes[0, 0].imshow(image); axes[0, 0].set_title(f'Input: {batch.metadata[SAMPLE_INDEX].sample_id}')
axes[0, 1].imshow(tissue_gt, cmap='tab10', vmin=0, vmax=len(TISSUE_CLASSES) - 1); axes[0, 1].set_title('Tissue — ground truth')
axes[0, 2].imshow(tissue_pred, cmap='tab10', vmin=0, vmax=len(TISSUE_CLASSES) - 1); axes[0, 2].set_title('Tissue — prediction')

axes[1, 0].imshow(image); axes[1, 0].set_title('Input')
axes[1, 1].imshow(image); _draw_nuclei(axes[1, 1], gt_centroids, gt_labels, gt_boxes)
axes[1, 1].set_title(f'Nuclei — ground truth ({len(gt_centroids)})')
axes[1, 2].imshow(image); _draw_nuclei(axes[1, 2], pred_centroids, pred_labels, pred_boxes)
axes[1, 2].set_title(f'Nuclei — prediction ({len(pred_centroids)} @ conf>{config.postprocess.confidence_threshold})')

for axis in axes.flat:
    axis.axis('off')
figure.tight_layout()
plt.show()

print(f'sample={batch.metadata[SAMPLE_INDEX].sample_id}  GT nuclei={len(gt_centroids)}  predicted={len(pred_centroids)}')

## 7. K-fold cross-validation (k=5) — train on all data

Trains 5 independent models: each uses 80% for training and the held-out 20% for validation, so **every image is validated exactly once**. Pretrained encoder, EMA, and the class-weighted loss are inherited from the §2 config (run §2 first). Reports mean ± std of the best per-fold metrics. Note: this runs the full `epochs` schedule five times.

In [ ]:
import copy, gc
import numpy as np
from prometheus.api import build_kfold_datamodule, build_model, build_trainer

NUM_FOLDS = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
KFOLD_DIR = Path('/content/drive/MyDrive/prometheus_runs/baseline_multitask_kfold')
KFOLD_MANIFEST = str(KFOLD_DIR / 'puma_kfold.json')
KFOLD_DIR.mkdir(parents=True, exist_ok=True)

fold_scores = []
for fold in range(NUM_FOLDS):
    print(f'\n===== Fold {fold + 1}/{NUM_FOLDS} =====')
    fold_config = copy.deepcopy(config)
    fold_config.paths.run_dir = str(KFOLD_DIR / f'fold_{fold}')

    train_loader, val_loader = build_kfold_datamodule(
        fold_config, fold_index=fold, num_folds=NUM_FOLDS, kfold_manifest_path=KFOLD_MANIFEST,
    )
    # Fresh model per fold; resume within a fold if it was interrupted.
    last_ckpt = Path(fold_config.paths.run_dir) / 'last.ckpt'
    fold_model = build_model(fold_config, pretrained=USE_PRETRAINED and not last_ckpt.exists())
    fold_trainer = build_trainer(fold_config, model=fold_model, datamodule=(train_loader, val_loader), device=device)
    fold_trainer.fit(resume_from=last_ckpt if last_ckpt.exists() else None)

    fold_scores.append({'fold': fold,
                        'nuclei_f1': fold_trainer.best_primary,
                        'tissue_dice': fold_trainer.best_tissue})
    print(f'Fold {fold}: best nuclei_f1={fold_trainer.best_primary:.4f}  best tissue_dice={fold_trainer.best_tissue:.4f}')

    del fold_model, fold_trainer, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()

f1 = np.array([s['nuclei_f1'] for s in fold_scores])
dice = np.array([s['tissue_dice'] for s in fold_scores])
print('\n===== 5-fold cross-validation summary =====')
for s in fold_scores:
    print(f"  fold {s['fold']}: nuclei_f1={s['nuclei_f1']:.4f}  tissue_dice={s['tissue_dice']:.4f}")
print(f'nuclei centroid F1 : {f1.mean():.4f} +/- {f1.std():.4f}')
print(f'tissue Dice        : {dice.mean():.4f} +/- {dice.std():.4f}')

## 6. Submission smoke test

Use the same checkpoint through the CLI so notebook and headless inference share one composition path:

```bash
prometheus predict --config configs/experiment/baseline_multitask.toml --checkpoint /path/to/best_primary.ckpt --input /path/to/sample.tif --output /content/prediction
```